# Gateway

The only code that builds a model client. Everything else receives `get_chat_model` as an argument
and never learns what a gateway is.

Two paths off the same workspace host: the AI Gateway serves the `finhive_router` and
`finhive_embeddings` services, and Databricks' own pay-per-token models answer on
`/serving-endpoints`, which is where the guardrails run because they are the most frequent and most
mechanical calls in the graph (`agent-design.md` §11.3). The host and the token both come from the
notebook context, so no workspace URL is written down and no PAT is stored.

Needs `langchain-openai` in the environment (§19.6 pins `langchain-openai==1.6.2`,
`langchain-core==1.6.3`, `openai==3.13.0`).

`check()` is defined but not called - it costs nine model calls, so `%run`ing this notebook must
stay free. Run `check()` in a cell to exercise it.

In [ ]:
%run ../config

In [ ]:
%run ./parsing

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# role -> (path, model). The only place a role becomes a model name.
ROLE_MODELS = {
    "router": (AI_GATEWAY_PATH, MODEL_ROUTER),
    "worker": (AI_GATEWAY_PATH, MODEL_ROUTER),
    "synthesizer": (AI_GATEWAY_PATH, MODEL_ROUTER),
    "guard_in": (SERVING_PATH, MODEL_GUARD_IN),
    "guard_out": (SERVING_PATH, MODEL_GUARD_OUT),
    "embedding": (AI_GATEWAY_PATH, MODEL_EMBEDDINGS),
}

# §21: reasoning models spent the whole output budget thinking and returned empty content in ~40%
# of calls. Floor every chat call.
MIN_OUTPUT_TOKENS = 300

_context = None


def _ctx():
    """Host and token both come from the notebook context, cached. No PAT, no hardcoded workspace."""
    global _context
    if _context is None:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        _context = (ctx.apiUrl().get().rstrip("/"), ctx.apiToken().get())
    return _context


def _chat(path, model, temperature, max_tokens):
    host, token = _ctx()
    return ChatOpenAI(
        model=model,
        base_url=f"{host}{path}",
        api_key=token,
        temperature=temperature,
        timeout=MODEL_TIMEOUT_SECONDS,
        # langchain-openai rewrites max_tokens to max_completion_tokens, which the gateway rejects
        # with 400 unknown field (§21). Pass the cap through untouched.
        extra_body={"max_tokens": max(max_tokens, MIN_OUTPUT_TOKENS)},
    )


def get_chat_model(role, temperature=None, max_tokens=None):
    if role not in ROLE_MODELS or role == "embedding":
        raise ValueError(f"unknown chat role {role!r}; known: "
                         f"{sorted(r for r in ROLE_MODELS if r != 'embedding')}")
    path, model = ROLE_MODELS[role]
    settings = ROLE_SETTINGS[role]
    return _chat(path, model,
                 settings["temperature"] if temperature is None else temperature,
                 max_tokens or settings["max_tokens"])


def get_embedding_model():
    path, model = ROLE_MODELS["embedding"]
    host, token = _ctx()
    # check_embedding_ctx_length would try to tokenize for an OpenAI model name that is not one
    return OpenAIEmbeddings(model=model, base_url=f"{host}{path}", api_key=token,
                            check_embedding_ctx_length=False)

In [ ]:
from pydantic import BaseModel, Field


class _Probe(BaseModel):
    """Flat, every field required - the shape §4.2 requires of every schema."""
    answer: str = Field(description="the capital city")
    confident: bool = Field(description="whether the answer is certain")


def check():
    """Nine live calls. Returns a result dict and raises on failure.

    Every chat role answers, embeddings return a vector, and structured output is probed three
    times: through the router service, and against each model the router routes to, directly. The
    third part is the one that matters - the router picks per request, so a single round trip
    through it exercises one of the two with 70/30 odds and would pass while the graph fails
    intermittently in production (agent-design.md §22, open items 10 and 11).
    """
    rows = []
    fallback = _Probe(answer="", confident=False)
    ask = ("Answer about world capitals.", "What is the capital of France?")

    for role in (r for r in ROLE_MODELS if r != "embedding"):
        try:
            text = message_text(get_chat_model(role).invoke(
                [{"role": "user", "content": "Reply with the single word: ready"}]))
            rows.append((role, bool(text.strip()), f"{ROLE_MODELS[role][1]} -> {text.strip()[:60]!r}"))
        except Exception as exc:
            rows.append((role, False, f"{type(exc).__name__}: {exc}"))

    try:
        vector = get_embedding_model().embed_query("What is Databricks?")
        rows.append(("embedding", len(vector) > 0, f"{MODEL_EMBEDDINGS} -> dim {len(vector)}"))
    except Exception as exc:
        rows.append(("embedding", False, f"{type(exc).__name__}: {exc}"))

    unsupported = check_schema_supported(_Probe)
    rows.append(("probe_schema", not unsupported,
                 f"unsupported={unsupported}" if unsupported else "flat, all required"))

    for label, chat in [("structured/router", get_chat_model("router"))] + [
        (f"structured/{model}", _chat(SERVING_PATH, model, 0.0, 600)) for model in MODELS_ROUTED
    ]:
        try:
            got = ask_structured(chat, *ask, _Probe, fallback)
            rows.append((label, bool(got.answer.strip()),
                         f"{got.answer!r} confident={got.confident}" if got.answer.strip()
                         else "fell back - response_format not honoured, use function calling"))
        except Exception as exc:
            rows.append((label, False, f"{type(exc).__name__}: {exc}"))

    for label, ok, detail in rows:
        print(f"{'PASS' if ok else 'FAIL'}  {label:<44}  {detail}")

    failed = [label for label, ok, _ in rows if not ok]
    if failed:
        raise RuntimeError(f"llm/gateway check failed: {failed}")
    return {"ok": True, "rows": rows}